<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# Communication Between Containers

Containers provide separate application environments, but applications often need to cooperate by exchanging data. A camera driver produces images that another program processes; a web server receives requests from a client.

The previous notebook grouped Duckiedrone containers into stacks, which tells us how services are deployed, but how do these programs communicate across their container boundaries?

In this notebook, we will connect two practice containers, inspect the network configuration that makes their connection possible, and compare it with a real service running on a Duckiedrone.

## Example: Following an image from the camera

The container names become easier to interpret when connected to a specific task. [Figure 1](#figure-1) shows the camera-data path.

<figure id="figure-1" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    camera
      |
      | capture images
      v
    driver-camera (stack: duckietown/duckiedrone)
      |
      | publish camera data
      v
    dtps (stack: duckietown/duckiedrone)
      |
      | transfer data to the bridge
      v
    ros2-camera (stack: ros2/duckiedrone)
      |
      | publish ROS 2 messages
      v
    ROS 2 application
  </pre>
  <figcaption>
    Figure 1: Camera data passes through services in two stacks before reaching a ROS 2 application.
  </figcaption>
</figure>

The `driver-camera` container runs the code that captures images. To understand settings such as exposure or resolution, inspect the driver's configuration and how the container starts. Later in this notebook, we will trace that startup configuration from the Compose file to the image's launcher.

DTPS carries camera and other sensor messages between participating services. It runs in its own `dtps` container. See the [DTPS middleware documentation](https://docs.duckietown.com/ente/duckietown-manual/04-software-tools/duckietown-postal-service-dtps.html?highlight=dtps) for details.

The `ros2-camera` container then bridges the camera data into ROS 2 topics (which we will learn more about in the dedicated [ROS 2 LX](https://github.com/duckietown/lx-dd-middleware-ros2)). The `zenoh-router` service supports communication among the ROS 2 containers; it is omitted from this simplified diagram. See the [ROS 2 documentation](https://www.ros.org/) for its messaging model.

The ToF distance sensor follows a similar pattern through `driver-tof-bottom`, `dtps`, and `ros2-tof-bottom` (`-bottom` for the downward-facing sensor). See the manual's [software architecture](https://docs.duckietown.com/ente/opmanual-dd24/60-software-architecture/architecture-diagram.html) for additional detail.

On a physical Duckiedrone, PX4 flight-control firmware (which is not containerized) runs on the flight-controller board. The `ros2-mavros` container exchanges [Micro Air Vehicle Link (MAVLink)](https://mavlink.io/) messages with that controller and makes its state and telemetry available to ROS 2 programs. On a virtual Duckiedrone instead, `dt-px4` runs PX4 in a container on the base station. The browser-based Dashboard receives ROS 2 data through `ros2-rosbridge-websocket`. See the [PX4 flight-controller documentation](https://docs.px4.io/main/) for details about the firmware.

### Shared files are one communication mechanism

In the previous [volumes notebook](./7-docker-volumes-bind-mounts-and-cleanup.ipynb), we showed how two containers shared data by accessing the same named volume. One wrote a file, and the other read it. This is useful when applications exchange files or need persistent storage. However, a program requesting a camera image does not necessarily need another program to save that image to a shared file first.

Applications can also communicate through __network sockets__. A server listens at an address and port, and a client connects to it using an agreed protocol, such as HTTP. Each application can keep its own filesystem. Docker provides [networking capabilities](https://docs.docker.com/engine/network/), while the applications define the requests and responses exchanged over a connection. 

### Connect two containers through a Docker network

We will reuse and build upon the web application from the image-building notebook. Previously, a client on the base station reached it through a published host port. This time, the client will run in another container.

Run these commands from the LX repository root, using the local practice daemon established earlier.

Rebuild the exercise image if it was removed during cleanup:

```bash
docker build --tag lx-docker-hello:0.1 packages/docker_exercises
```

Create a network and start the server on it:

```bash
docker network create lx-docker-net

docker run --detach --name lx-docker-network-web \
  --network lx-docker-net \
  lx-docker-hello:0.1
```

The network uses Docker's `bridge` driver. See Docker's [bridge-network guide](https://docs.docker.com/engine/network/drivers/bridge/) for details. On a __user-defined bridge network__, Docker provides name resolution so containers can find one another by name. 

Notice that this command has neither `--mount` nor `--publish`.

Start a temporary client:

```bash
docker run --rm --name lx-docker-network-client \
  --network lx-docker-net \
  alpine:3.21 \
  wget -T 5 -qO- http://lx-docker-network-web:8080/
```

`wget` makes an HTTP request and prints the response body. Docker resolves `lx-docker-network-web` to the server container's address.

You should receive the application response from the earlier notebook. If the request fails, inspect the server:

```bash
docker container ls --all --filter 'name=lx-docker-network-web'
docker logs lx-docker-network-web
```

If the server was still starting, repeat the request after it is ready. The successful request demonstrates communication between two containers without a shared directory or a published host port.

### Why does `localhost` give a different result?

Repeat the request, changing only its destination:

```bash
docker run --rm --name lx-docker-network-client \
  --network lx-docker-net \
  alpine:3.21 \
  wget -T 5 -qO- http://127.0.0.1:8080/
```

This request should fail: the temporary client container has no server listening on its own port `8080`. In this network configuration, each container has its own network namespace. Therefore, `127.0.0.1` refers to the container making the request. Joining the same Docker network does not make the containers share `localhost`.

Inspect the network:

```bash
docker network inspect lx-docker-net
```

Find the server under `Containers`. Note that the temporary client is no longer listed because its command finished and `--rm` removed it.

### Listening, exposing, and publishing are different

Three settings can appear to describe the same port, but they do different jobs:

[Table 1](#table-1) summarizes these cases.

<table id="table-1" class="lx-table">
  <caption>Table 1: Listening, exposing, and publishing a container port.</caption>
  <thead>
    <tr>
      <th>Setting</th>
      <th>What it does</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>The application listens on <code>0.0.0.0:8080</code></td>
      <td>The running program accepts connections on port <code>8080</code> through its network interfaces.</td>
    </tr>
    <tr>
      <td>The Dockerfile contains <code>EXPOSE 8080</code></td>
      <td>The image documents an intended application port. This does not start a server or publish the port.</td>
    </tr>
    <tr>
      <td>The container starts with <code>--publish 127.0.0.1:8088:8080</code></td>
      <td>Docker creates a mapping from the host's loopback port <code>8088</code> to the container's port <code>8080</code>.</td>
    </tr>
  </tbody>
</table>

The image's `EXPOSE` instruction [is metadata](https://docs.docker.com/reference/dockerfile/#expose); it does not enforce access rules. Containers on the same ordinary user-defined bridge network can reach one another's listening ports without `EXPOSE` or `--publish`.

Check the practice server's published mappings:

```bash
docker container port lx-docker-network-web
```

This should print no mappings. Nevertheless, the other container successfully reached the server.

Publishing provides a host-side entry point, as in our earlier `127.0.0.1:8088` exercise. It is unnecessary for this connection within the Docker network. See Docker's [port-publishing documentation](https://docs.docker.com/engine/network/port-publishing/).

### Where are the boundaries defined?

A container's interface is determined by several cooperating settings:

- __The image__ provides the program, dependencies, and default startup configuration.

- __The runtime configuration__ selects networks, mounts, published ports, and other resources. These settings can be supplied through `docker run` or a Compose file.

- __The application__ chooses where to listen, which protocol to speak, and which requests to accept.

For example, Docker can make a camera service reachable, but the application's protocol and access controls determine which operations a client may perform. Similarly, placing two containers on one bridge network normally permits connections to their listening ports; it does not grant access to only one selected API operation.

### Read a real Duckiedrone configuration

We can now examine the Docker settings behind a Duckiedrone service without needing to operate physical sensors.

Open the [Duckiedrone Compose file](https://github.com/duckietown/duckietown-shell-commands/blob/312fa45782ec700c3e27c62dc193c8bcc22439f8/stack/stacks/duckietown/duckiedrone.yaml).

Find the `dtps` service. This excerpt shows its communication settings, with other fields omitted:

```yaml
services:
  dtps:
    network_mode: host
    command: --tcp-port 11911 --tcp-host 0.0.0.0 --unix-path /dtps/switchboard.sock
    volumes:
      - /data/ramdisk/dtps:/dtps
```

Read this as configuration evidence:

1. The command asks DTPS to provide a TCP endpoint on port `11911`.

2. It also asks DTPS to create a Unix socket at `/dtps/switchboard.sock`.

3. A bind mount makes the host directory `/data/ramdisk/dtps` available inside the container as `/dtps`.

4. `network_mode: host` selects a different networking arrangement from our practice bridge.

#### Host networking changes the network boundary

On a Linux Docker host, `network_mode: host` makes the container share the host's network namespace.

Consequently, the service uses the host's network addresses and ports directly. A separate Docker port-publishing rule is unnecessary, and `localhost` refers to the shared host network namespace.

This explains why the absence of a `ports:` section does not imply that this service is unreachable. Its reachability also depends on the application's listening address and the [host's network controls](https://docs.docker.com/engine/network/drivers/host/).  

#### A shared directory can contain a communication endpoint

A __Unix-domain socket__ lets processes on the same host communicate through an endpoint identified by a filesystem path. The socket path resembles a filename, but it is not an ordinary file containing saved messages. Processes connect to the socket and exchange data through the kernel.

Now find the `driver-camera` service in the same Compose file. It also mounts:

```yaml
volumes:
  - /data/ramdisk/dtps:/dtps
```

Both containers therefore see the same underlying directory at `/dtps`. This makes the DTPS socket path available to the camera container (subject to filesystem permissions and application configuration).

### Follow the camera service from image to startup

In the camera service definition, find:

```yaml
image: ${REGISTRY}/duckietown/dt-duckiebot-interface:ente-${ARCH}
environment:
  DT_LAUNCHER: sensor-camera
```

The `image` field selects the software package. The environment variable selects the camera launcher when that package runs.

Open the image's [camera-image Dockerfile](https://github.com/duckietown/dt-duckiebot-interface/blob/ente/Dockerfile) and locate these instructions:

```dockerfile
COPY ./packages "${PROJECT_PATH}/packages"
COPY ./launchers/. "${PROJECT_LAUNCHERS_PATH}/"
CMD ["bash", "-c", "dt-launcher-${DT_LAUNCHER}"]
```

The Dockerfile copies the application packages and launcher scripts into the image. Its default command uses `DT_LAUNCHER` to choose what to start.

Compare this with `driver-tof-bottom` in the Compose file: it selects the same image but a different launcher.

This is a concrete example of the distinction between an image and a container. One image can supply several programs; runtime configuration determines which program a particular container starts and which host resources it receives.

The Dockerfile link follows the `ente` branch and can change. If its contents differ, use the repository history to identify the version you are inspecting.

## Verify configuration on a running deployment

Start and connect to your virtual or physical Duckiedrone:

- Boot the physical Duckiedrone following its setup instructions, then open an SSH session on it.

- For an existing virtual Duckiedrone, use `dts duckiebot virtual start DUCKIEDRONE_NAME`, then `dts duckiebot virtual connect DUCKIEDRONE_NAME`.

Inside the Duckiedrone shell, run `docker ps` to check whether the `dtps` container is running. Then, run the following commands to inspect its configuration:

```bash
docker container inspect dtps \
  --format 'Network mode: {{.HostConfig.NetworkMode}}'

docker container inspect dtps \
  --format '{{range .Mounts}}{{println .Type .Source "->" .Destination}}{{end}}'

docker container inspect dtps \
  --format 'Command: {{json .Config.Cmd}}'
```

Compare the network mode, mount paths, and command with the source excerpt.

`docker container inspect` also works for stopped containers. If the `dtps` container is missing, check the deployment setup instructions. 

## Clean up the practice resources

Leave the Duckiedrone shell with `exit`. Back in the base-station terminal, verify your local practice Docker connection, then remove the server and network created in this notebook:

```bash
docker stop lx-docker-network-web
docker container rm lx-docker-network-web
docker network rm lx-docker-net
```

If you rebuilt the exercise image solely for this activity, remove it too:

```bash
docker image rm lx-docker-hello:0.1
```

If you started the virtual Duckiedrone for this activity and no other work depends on it, stop it from the base-station terminal:

```bash
dts duckiebot virtual stop DUCKIEDRONE_NAME
```

## Further reading

The [Duckiedrone container reference](https://docs.duckietown.com/ente/opmanual-dd24/60-software-architecture/containers-running-on-the-duckiedrone.html) lists the current stacks and services. The [software architecture](https://docs.duckietown.com/ente/opmanual-dd24/60-software-architecture/architecture-diagram.html) explains their data paths.

## Checkpoint

Before running the self-check, consider these questions:

1. What evidence showed that two containers communicated without sharing files?

2. Why did the request to `127.0.0.1:8080` fail inside the client container?

3. Does `EXPOSE 8080` prevent connections to other ports?

4. Why can the DTPS service accept network connections without a Compose `ports:` mapping?

5. What is the difference between mounting a directory containing saved images and mounting a directory containing a Unix socket?

6. Which camera-service settings come from the Dockerfile, and which come from Compose?

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()


## Summary and conclusions

Two containers exchanged an HTTP request and response over a Docker network without shared files or published host ports. The Duckiedrone example showed another arrangement: host networking and a shared Unix-socket directory. The image supplies software, runtime settings provide connections, and the application defines how to use them. Next, [use Docker contexts to select the daemon](./10-docker-contexts-and-local-targets.ipynb) for an inspection command.
